# Sellers Data Cleaning

Problems to Investigate
- Check for any cities associated with multiple states
- Any whitespace

In [1]:
import pandas as pd 

sellers_raw = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
sellers = sellers_raw.copy()

print("Original Shape:", sellers_raw.shape)
print("Working Shape:", sellers.shape)

Original Shape: (3095, 4)
Working Shape: (3095, 4)


In [2]:
# Check for any cities associated with multiple states

city_state_counts = (
    sellers.groupby("seller_city")["seller_state"]
    .nunique()
)

print(
    "Cities associated with multiple states:",
    (city_state_counts > 1).sum()
)

Cities associated with multiple states: 23


In [5]:
# Identify cities associated with multiple states
inconsistent_cities = (
    sellers.groupby("seller_city")["seller_state"]
    .nunique()
)

inconsistent_cities = inconsistent_cities[
    inconsistent_cities > 1
]

# Display city/state combinations
city_state_combinations = (
    sellers[
        sellers["seller_city"].isin(inconsistent_cities.index)
    ][["seller_city", "seller_state"]]
    .drop_duplicates()
    .sort_values(["seller_city", "seller_state"])
)

display(city_state_combinations)

,seller_city,seller_state
1825,andradas,MG
1656,andradas,SP
25,belo horizonte,MG
549,belo horizonte,SP
255,blumenau,SC
2455,blumenau,SP
143,caxias do sul,RS
70,caxias do sul,SP
1084,chapeco,SC
1552,chapeco,SP


In [4]:
# Check for whitespace

string_columns = sellers.select_dtypes(include=["object", "str"]).columns

for column in string_columns:
    count = sellers[column].str.contains(
        r"^\s|\s$", na=False
    ).sum()

    print(f"{column}: {count} values with whitespace")

sellers.columns = (
    sellers.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(sellers.columns.tolist())

seller_id: 0 values with whitespace
seller_city: 0 values with whitespace
seller_state: 0 values with whitespace
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [6]:
sellers.to_csv(
    "../data/processed/olist_sellers_cleaned.csv",
    index=False
)